# dashboard objective loss integration v1

目标：验证 `src/dashboard` 可以通过 `analysis_manifest.json` 读取 `objective.loss.v1` sidecar，并把它和 objective score、promotion score 一起写入统一 dashboard export。

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd

REPO = Path('/Users/rziiiii/dev/workspace')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.dashboard import load_model_matrix_dashboard, write_model_matrix_dashboard_export

CARD = Path('research/dashboard_objective_loss_integration_v1')
MATRIX_CARD = CARD / 'matrix_card'
EXPORT_PATH = CARD / 'dashboard_export.json'
SOURCE_LOSS_PATH = Path('research/objective_loss_contract_v1/objective_losses.json')

CARD.mkdir(parents=True, exist_ok=True)
MATRIX_CARD.mkdir(parents=True, exist_ok=True)
{
    'card': str(CARD),
    'matrix_card': str(MATRIX_CARD),
    'source_loss_exists': SOURCE_LOSS_PATH.exists(),
    'dashboard_export': str(EXPORT_PATH),
}

{'card': 'research/dashboard_objective_loss_integration_v1',
 'matrix_card': 'research/dashboard_objective_loss_integration_v1/matrix_card',
 'source_loss_exists': True,
 'dashboard_export': 'research/dashboard_objective_loss_integration_v1/dashboard_export.json'}

## 构造 manifest card

本实验只构造 dashboard 需要的最小公开 sidecar。`objective_losses.json` 复制自 `objective_loss_contract_v1` 的稳定 contract 输出，然后由本 card 的 `analysis_manifest.json` 显式暴露。

In [2]:
objective_losses = json.loads(SOURCE_LOSS_PATH.read_text(encoding='utf-8'))
assert objective_losses['schema_version'] == 'objective.loss.v1'
assert objective_losses['reward_surface'] == 'retired'

summary = {
    'schema_version': 'analysis.report.v1',
    'status': 'dashboard_objective_loss_integration_completed',
    'model_count': 2,
    'objective_loss_schema_version': objective_losses['schema_version'],
    'note': '中文 review：dashboard export 通过 manifest 读取 objective loss。',
}
(MATRIX_CARD / 'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
(MATRIX_CARD / 'objective_losses.json').write_text(
    json.dumps(objective_losses, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

pd.DataFrame([
    {
        'rank': 1,
        'model': 'ridge_mse__loss_contract_smoke',
        'base_model': 'ridge_mse',
        'feature_set': 'loss_contract_smoke',
        'return_pct': 2.4,
        'relative_return_pct': 0.9,
        'stability_score': 2.1,
    },
    {
        'rank': 2,
        'model': 'gcn_mse__loss_contract_smoke',
        'base_model': 'gcn_mse',
        'feature_set': 'loss_contract_smoke',
        'return_pct': -0.8,
        'relative_return_pct': -1.2,
        'stability_score': -0.4,
    },
]).to_csv(MATRIX_CARD / 'model_comparison.csv', index=False)
pd.DataFrame([
    {'model': 'ridge_mse__loss_contract_smoke', 'regime': 'bull', 'return_pct': 2.4},
    {'model': 'gcn_mse__loss_contract_smoke', 'regime': 'stress', 'return_pct': -0.8},
]).to_csv(MATRIX_CARD / 'regime_metrics.csv', index=False)
pd.DataFrame([
    {'model': 'ridge_mse__loss_contract_smoke', 'window': 'smoke', 'return_pct': 2.4},
    {'model': 'gcn_mse__loss_contract_smoke', 'window': 'smoke', 'return_pct': -0.8},
]).to_csv(MATRIX_CARD / 'window_metrics.csv', index=False)
pd.DataFrame([
    {
        'schema_version': 'objective.evaluation.v1',
        'model': 'ridge_mse__loss_contract_smoke',
        'mse': objective_losses['evaluation']['mse'],
        'classification_accuracy': objective_losses['evaluation']['classification_accuracy'],
        'ic': objective_losses['evaluation']['ic'],
        'rank_ic': objective_losses['evaluation']['rank_ic'],
    }
]).to_csv(MATRIX_CARD / 'objective_scores.csv', index=False)
pd.DataFrame([
    {
        'schema_version': 'objective.promotion_score.v1',
        'rank': 1,
        'model': 'ridge_mse__loss_contract_smoke',
        'promotion_score': 2.1,
        'promotion_decision': 'research_only',
        'promotion_blockers': 'dashboard loss integration is framework evidence, not strategy promotion',
    }
]).to_csv(MATRIX_CARD / 'promotion_scores.csv', index=False)
pd.DataFrame([{'model': 'ridge_mse__loss_contract_smoke', 'prediction': 0.1}]).to_csv(
    MATRIX_CARD / 'predictions_private.csv',
    index=False,
)

manifest = {
    'artifact_id': 'dashboard_objective_loss_integration_v1',
    'artifact_type': 'analysis_report',
    'producer': 'src.dashboard.model_matrix',
    'schema_version': 'artifact.v1',
    'paths': {
        'summary': 'summary.json',
        'model_comparison': 'model_comparison.csv',
        'regime_metrics': 'regime_metrics.csv',
        'window_metrics': 'window_metrics.csv',
        'objective_scores': 'objective_scores.csv',
        'promotion_scores': 'promotion_scores.csv',
        'objective_losses': 'objective_losses.json',
        'predictions': 'predictions_private.csv',
    },
    'metadata': {'analysis_report_schema_version': 'analysis.report.v1'},
    'manifest_hash': 'l' * 64,
}
(MATRIX_CARD / 'analysis_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
sorted(path.name for path in MATRIX_CARD.iterdir())

['analysis_manifest.json',
 'model_comparison.csv',
 'objective_losses.json',
 'objective_scores.csv',
 'predictions_private.csv',
 'promotion_scores.csv',
 'regime_metrics.csv',
 'summary.json',
 'window_metrics.csv']

## 读取 dashboard payload

这里验证 dashboard loader 不靠固定文件名猜测，而是从 manifest 中找到 `objective_losses`，并且继续隐藏 private predictions。

In [3]:
payload = load_model_matrix_dashboard(MATRIX_CARD, top_n=2)
assert payload['manifest_source']['kind'] == 'analysis_manifest'
assert payload['objective']['scores'][0]['schema_version'] == 'objective.evaluation.v1'
assert payload['objective']['promotion_scores'][0]['schema_version'] == 'objective.promotion_score.v1'
assert payload['objective']['losses']['schema_version'] == 'objective.loss.v1'
assert payload['objective']['losses']['reward_surface'] == 'retired'
assert payload['artifacts']['objective_losses'].endswith('objective_losses.json')
assert 'predictions' not in payload['artifacts']
{
    'artifact_id': payload['artifact_id'],
    'objective_loss_schema': payload['objective']['losses']['schema_version'],
    'loss_component_keys': [key for key in ('mse_loss', 'classification_loss', 'ic_loss', 'ranking_loss') if key in payload['objective']['losses']],
    'artifact_keys': sorted(payload['artifacts']),
}

{'artifact_id': 'dashboard_objective_loss_integration_v1',
 'objective_loss_schema': 'objective.loss.v1',
 'loss_component_keys': ['mse_loss',
  'classification_loss',
  'ic_loss',
  'ranking_loss'],
 'artifact_keys': ['analysis_manifest',
  'model_comparison',
  'objective_losses',
  'objective_scores',
  'promotion_scores',
  'regime_metrics',
  'summary',
  'window_metrics']}

## 写出 dashboard export

`write_model_matrix_dashboard_export` 应保留同一份 `objective.loss.v1`，供 dashboard 页面或后续 export 消费。

In [4]:
export_payload = write_model_matrix_dashboard_export(MATRIX_CARD, output_path=EXPORT_PATH, top_n=2)
saved = json.loads(EXPORT_PATH.read_text(encoding='utf-8'))
assert saved['schema_version'] == 'dashboard.model_matrix.v1'
assert saved['objective']['losses'] == payload['objective']['losses']
assert saved['artifacts']['objective_losses'].endswith('objective_losses.json')

run_summary = {
    'schema_version': 'research.dashboard_objective_loss_integration.v1',
    'status': 'completed',
    'matrix_card': str(MATRIX_CARD),
    'dashboard_export': str(EXPORT_PATH),
    'payload_schema_version': saved['schema_version'],
    'objective_loss_schema_version': saved['objective']['losses']['schema_version'],
    'metric_source_schema_version': saved['objective']['losses']['metric_source_schema_version'],
    'reward_surface': saved['objective']['losses']['reward_surface'],
    'optimization_direction': saved['objective']['losses']['optimization_direction'],
    'loss_component_keys': [key for key in ('mse_loss', 'classification_loss', 'ic_loss', 'ranking_loss') if key in saved['objective']['losses']],
    'rank_group_count': saved['objective']['losses'].get('rank_group_count'),
    'artifact_keys': sorted(saved['artifacts']),
    'private_predictions_hidden': 'predictions' not in saved['artifacts'],
    'conclusion': 'src/dashboard 已把 objective.loss.v1 纳入 manifest-first dashboard export；reward surface 继续保持 retired。',
}
(CARD / 'analysis_summary.json').write_text(json.dumps(run_summary, ensure_ascii=False, indent=2), encoding='utf-8')
run_summary

{'schema_version': 'research.dashboard_objective_loss_integration.v1',
 'status': 'completed',
 'matrix_card': 'research/dashboard_objective_loss_integration_v1/matrix_card',
 'dashboard_export': 'research/dashboard_objective_loss_integration_v1/dashboard_export.json',
 'payload_schema_version': 'dashboard.model_matrix.v1',
 'objective_loss_schema_version': 'objective.loss.v1',
 'metric_source_schema_version': 'objective.evaluation.v1',
 'reward_surface': 'retired',
 'optimization_direction': 'minimize',
 'loss_component_keys': ['mse_loss',
  'classification_loss',
  'ic_loss',
  'ranking_loss'],
 'rank_group_count': 3,
 'artifact_keys': ['analysis_manifest',
  'model_comparison',
  'objective_losses',
  'objective_scores',
  'promotion_scores',
  'regime_metrics',
  'summary',
  'window_metrics'],
 'private_predictions_hidden': True,
 'conclusion': 'src/dashboard 已把 objective.loss.v1 纳入 manifest-first dashboard export；reward surface 继续保持 retired。'}

## 中文结论

- 做了什么：验证 `src/dashboard` 通过 `analysis_manifest.json` 读取 `objective_losses.json`，并把 `objective.loss.v1` 与 `objective.evaluation.v1`、`objective.promotion_score.v1` 一起写入 `dashboard_export.json`。
- 核心指标：export 中 `objective.losses.schema_version=objective.loss.v1`、`reward_surface=retired`、`optimization_direction=minimize`，并保留 `mse_loss`、`classification_loss`、`ic_loss`、`ranking_loss` 和 `rank_group_count`；private predictions 没有进入 artifacts。
- regime 差异：本实验只用 bull/stress 的 smoke regime 行验证 dashboard 数据通路，不做策略优劣结论；真实区间差异仍由 `src/analysis` 和完整模型矩阵 notebook 承接。
- 是否晋升：这是框架能力晋升到 `src/dashboard` 的证据，不是交易策略晋升；promotion sidecar 标记为 `research_only`。
- 风险：当前只把 loss 放入 dashboard export，还没有扩展页面卡片的 loss 专用展示组件；后续需要决定页面如何并列展示 loss、score 和 promotion。
- 下一步：让 QuantBench model matrix notebook 在生成 objective score/promotion score 时同步写出 `objective_losses.json`，dashboard 页面再增加 objective loss 对比区域。